# ================================================================
#  PET Late-Fusion Ensemble — Single-Image Inference (Soft Vote)
# ================================================================

In [25]:
import json, numpy as np, torch, torch.nn as nn, pandas as pd
import torch.nn.functional as F
import nibabel as nib
from pathlib import Path

## test image

In [31]:
IMAGE_ID   = "I193095"                # AMYLOID NEGATIVE EXAMPLE
TRUE_LABEL = 0                                   # 0=Negative

#IMAGE_ID   = "I225450"                # AMYLOID POSITIVE EXAMPLE
#TRUE_LABEL = 1                                   # 0=Positive

## architecture (from training notebook)

In [32]:
class Simple3DCNN_LateFusion(nn.Module):
    def __init__(self, channels=(32,64,128,256), fc_units=512,
                 dropout=0.3, kernel_size=3, demo_hidden=16):
        super().__init__()
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks += [
                nn.Conv3d(in_ch, out_ch, kernel_size=kernel_size,
                          padding=kernel_size//2, bias=False),
                nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
                nn.MaxPool3d(2),
            ]
            in_ch = out_ch
        self.encoder     = nn.Sequential(*blocks)
        self.gap         = nn.AdaptiveAvgPool3d(1)
        self.demo_branch = nn.Sequential(nn.Linear(2, demo_hidden), nn.ReLU())
        self.dropout     = nn.Dropout(dropout)
        self.fc          = nn.Linear(in_ch + demo_hidden, fc_units)
        self.head        = nn.Linear(fc_units, 2)

    def forward(self, img, demo):
        x = self.gap(self.encoder(img)).flatten(1)
        x = self.dropout(torch.cat([x, self.demo_branch(demo)], dim=1))
        return self.head(F.relu(self.fc(x)))

# load models

In [33]:
# ── CONFIG ──────────────────────────────────────────────────────
FOLD_DIR   = Path("models")
IMAGE_DIR  = Path("images")
CSV_PATH   = Path("data/amy_dataset_final.csv")  # columns: image_id, AGE_AT_SCAN, SEX

# ────────────────────────────────────────────────────────────────

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FOLDS = 7
CFG     = {"channels": (32, 64, 128, 256), "kernel_size": 3, "fc_units": 128,
           "dropout": 0.3, "demo_hidden": 16}

# ── Load subject from CSV ────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df["image_id_2_str"] = df["image_id_2_str"].astype(str).str.replace(".nii.gz", "", regex=False)
row = df[df["image_id_2_str"] == IMAGE_ID.replace(".nii.gz", "")].iloc[0]

AGE = float(row["AGE_AT_SCAN"])
SEX = str(row["SEX"])
AGE_MEAN, AGE_STD = float(df["AGE_AT_SCAN"].mean()), float(df["AGE_AT_SCAN"].std())
age_z = (AGE - AGE_MEAN) / AGE_STD
sex_f = 1.0 if SEX.strip().upper() in ("M", "MALE", "1") else 0.0
demo  = torch.tensor([[age_z, sex_f]], dtype=torch.float32).to(DEVICE)
print(f"Subject: {IMAGE_ID}  AGE={AGE}  SEX={SEX}")

# ── Load image ───────────────────────────────────────────────────
img_np = nib.load(str(IMAGE_DIR / f"{IMAGE_ID}.nii.gz")).get_fdata(dtype=np.float32)
x      = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).to(DEVICE)


Subject: I193095  AGE=75.0  SEX=F


## Ensemble inference

In [34]:
fold_probs, fold_thresholds = [], []

for k in range(N_FOLDS):
    ckpt = torch.load(FOLD_DIR / f"fold_{k}.pt", map_location=DEVICE, weights_only=False)
    thr  = ckpt["best_threshold"]

    model = Simple3DCNN_LateFusion(**CFG).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    with torch.no_grad():
        prob = torch.softmax(model(x, demo), dim=1)[0, 1].item()

    fold_probs.append(prob)
    fold_thresholds.append(thr)
    print(f"  Fold {k}: prob={prob:.3f}  thr={thr:.3f}  → {'POSITIVE' if prob >= thr else 'negative'}")
    del model; torch.cuda.empty_cache()

  Fold 0: prob=0.171  thr=0.470  → negative
  Fold 1: prob=0.056  thr=0.400  → negative
  Fold 2: prob=0.071  thr=0.560  → negative
  Fold 3: prob=0.074  thr=0.570  → negative
  Fold 4: prob=0.077  thr=0.400  → negative
  Fold 5: prob=0.363  thr=0.600  → negative
  Fold 6: prob=0.294  thr=0.520  → negative


## Soft vote

In [35]:
avg_prob   = float(np.mean(fold_probs))
mean_thr   = float(np.mean(fold_thresholds))
prediction = int(avg_prob >= mean_thr)

print("\n" + "="*50)
print(f"  Ensemble avg prob : {avg_prob:.3f}")
print(f"  Mean threshold    : {mean_thr:.3f}")
print(f"  Prediction        : {'AMYLOID POSITIVE (1)' if prediction else 'AMYLOID NEGATIVE (0)'}")
if TRUE_LABEL is not None:
    print(f"  True label        : {'AMYLOID POSITIVE (1)' if TRUE_LABEL else 'AMYLOID NEGATIVE (0)'}")
    print(f"  Result            : {'✓ CORRECT' if prediction == TRUE_LABEL else '✗ WRONG'}")
print("="*50)


  Ensemble avg prob : 0.158
  Mean threshold    : 0.503
  Prediction        : AMYLOID NEGATIVE (0)
  True label        : AMYLOID NEGATIVE (0)
  Result            : ✓ CORRECT
